In [ ]:
import os
import json
import time
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from datetime import datetime
import random

# 📌 Ancoragem absoluta na raiz do repositório
try:
    BASE_DIR = Path(__file__).resolve().parent.parent.parent.parent.parent
except NameError:
    # Fallback para execução interativa no Notebook
    BASE_DIR = Path.cwd()

PASTA_DESTINO = BASE_DIR / "data" / "raw" / "sigaa"
CAMINHO_ARQUIVO = PASTA_DESTINO / "materias.json"


def limpar_nome_departamento(texto_bruto: str) -> str:
    """
    Trata o texto da opção selecionada no select de unidades.
    Exemplo entrada: "DEPTO ADMINISTRAÇÃO - BRASÍLIA - 11.01.01.02.01"
    Exemplo saída: "DEPTO ADMINISTRAÇÃO"
    """
    if not texto_bruto:
        return ""
    partes = texto_bruto.split(" - ")
    return partes[0].strip()


def salvar_lote_incremental(novos_itens: list, caminho_arquivo: Path):
    """
    Salva novos itens no JSON em disco preservando o conteúdo que já existia.
    """
    if not novos_itens:
        return

    dados_existentes = []
    
    # Garante a criação da estrutura de pastas na raiz
    caminho_arquivo.parent.mkdir(parents=True, exist_ok=True)

    if caminho_arquivo.exists() and caminho_arquivo.stat().st_size > 0:
        try:
            with open(caminho_arquivo, "r", encoding="utf-8") as f:
                dados_existentes = json.load(f)
        except Exception:
            dados_existentes = []

    dados_existentes.extend(novos_itens)

    with open(caminho_arquivo, "w", encoding="utf-8") as f:
        json.dump(dados_existentes, f, ensure_ascii=False, indent=4)

    print(f"  💾 [BACKUP] Lote de {len(novos_itens)} itens salvo no JSON em disco!\nCaminho: {caminho_arquivo}\n")


def generate_seed_departamentos():
    num = 0

    inicio_tempo = time.time()
    hora_inicio = datetime.now().strftime("%H:%M:%S")

    print(f"🚀 Processo iniciado às: {hora_inicio}")

    # Mapeamento prévio para pular matérias já gravadas
    codigos_processados = set()
    if CAMINHO_ARQUIVO.exists() and CAMINHO_ARQUIVO.stat().st_size > 0:
        try:
            with open(CAMINHO_ARQUIVO, "r", encoding="utf-8") as file:
                materias_existentes = json.load(file)
                for m in materias_existentes:
                    if "codigo" in m and "nivel" in m:
                        # Adiciona como tupla imutável (codigo, nivel)
                        codigos_processados.add((m["codigo"], m["nivel"]))
            print(f"ℹ️ {len(codigos_processados)} combinações (código, nível) já existem no JSON.")
        except Exception as e:
            print(f"⚠️ Erro ao ler JSON: {type(e).__name__}")

    # Opções do Chrome com flags de estabilidade
    options = webdriver.ChromeOptions()
    # options.add_argument("--start-maximized")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--headless")

    driver = webdriver.Chrome(options=options)
    
    wait_geral = WebDriverWait(driver, 10)
    wait_rapido = WebDriverWait(driver, 2)

    # Buffer de salvamento temporário (salva no disco a cada 10 itens)
    buffer_lote = []
    
    url = "https://sigaa.unb.br/sigaa/public/componentes/busca_componentes.jsf"

    try:
        driver.get(url)

        wait_geral.until(EC.presence_of_element_located((By.ID, "form:unidades")))
        wait_geral.until(EC.presence_of_element_located((By.ID, "form:nivel")))

        select_element_1 = driver.find_element(By.ID, "form:nivel")
        select_element_2 = driver.find_element(By.ID, "form:unidades")

        opcoes_1 = Select(select_element_1).options
        opcoes_2 = Select(select_element_2).options

        print(f"Total de opções encontradas: {len(opcoes_1)} níveis | {len(opcoes_2)} departamentos")

        for n in range(len(opcoes_1)):
            for i in range(len(opcoes_2)):
                driver.get(url)
                wait_geral.until(EC.presence_of_element_located((By.ID, "form:unidades")))

                Select(driver.find_element(By.ID, "form:nivel")).select_by_index(n)
                select_unidades = Select(driver.find_element(By.ID, "form:unidades"))
                select_nivel = Select(driver.find_element(By.ID, "form:nivel"))

                opcao_depto = select_unidades.options[i]
                opcao_nivel = select_nivel.options[n]
                
                nome_depto_bruto = opcao_depto.text.strip()
                nivel = opcao_nivel.text.strip()
                depto_nome_limpo = limpar_nome_departamento(nome_depto_bruto)

                if not depto_nome_limpo or "SELECIONE" in depto_nome_limpo.upper():
                    continue

                print(f"\n--- Processando: {depto_nome_limpo} (Nível: {nivel}) ---")
                select_unidades.select_by_index(i)

                driver.find_element(By.ID, "form:btnBuscarComponentes").click()

                try:
                    wait_rapido.until(EC.presence_of_element_located((By.ID, "formListagemComponentes")))

                    linhas_tabela = driver.find_elements(
                        By.XPATH, "//form[@id='formListagemComponentes']//table/tbody/tr"
                    )
                    total_linhas = len(linhas_tabela)

                    for index in range(total_linhas):
                        try:
                            wait_geral.until(EC.presence_of_element_located((By.ID, "formListagemComponentes")))
                            tr = driver.find_elements(
                                By.XPATH, "//form[@id='formListagemComponentes']//table/tbody/tr"
                            )[index]

                            codigo_elem = tr.find_elements(By.XPATH, "./td[1]")
                            nome_elem = tr.find_elements(By.XPATH, "./td[2]")
                            link_elem = tr.find_elements(By.XPATH, "./td[5]//a")

                            if not (codigo_elem and nome_elem and link_elem):
                                continue

                            codigo = codigo_elem[0].text.strip().upper()
                            nome = nome_elem[0].text.strip().upper()
                            script_onclick = link_elem[0].get_attribute("onclick")

                            if not script_onclick:
                                continue

                            # Em vez de: if codigo in codigos_processados[index][0] ...
                            if (codigo, nivel) in codigos_processados:
                                print(f"  ⏩ Materia {codigo} ({nivel}) já processada. Pulando...")
                                continue

                            driver.execute_script(script_onclick)

                            wait_geral.until(EC.presence_of_element_located((By.CLASS_NAME, "visualizacao")))

                            pre_req, co_req, equiva = [], [], []
                            ementa = ""
                            carga_materia = "0"

                            linhas_detalhes = driver.find_elements(
                                By.XPATH, "//table[@class='visualizacao']//tbody//tr"
                            )

                            for row in linhas_detalhes:
                                texto_linha = row.text

                                if "Pré-Requisitos:" in texto_linha:
                                    tds = row.find_elements(By.TAG_NAME, "td")
                                    for text in tds:
                                        pre_req.append(text.text)

                                elif "Co-Requisitos:" in texto_linha:
                                    tds = row.find_elements(By.TAG_NAME, "td")
                                    for text in tds:
                                        co_req.append(text.text)

                                elif "Equivalências:" in texto_linha:
                                    tds = row.find_elements(By.TAG_NAME, "td")
                                    for text in tds:
                                        equiva.append(text.text)

                                elif "Ementa:" in texto_linha or "Ementa/Descrição:" in texto_linha:
                                    tds = row.find_elements(By.TAG_NAME, "td")
                                    for text in tds:
                                        ementa = text.text

                                elif "Carga Horária Total:" in texto_linha or "Total de Carga Horária" in texto_linha:
                                    tds = row.find_elements(By.TAG_NAME, "td")
                                    for text in tds:
                                        texto_ch = text.text.strip()
                                        carga_materia = "".join(filter(str.isdigit, texto_ch)) or "0"

                            ch_int = int(carga_materia)

                            materia_dados = {
                                "nivel": nivel,
                                "codigo": codigo,
                                "nome": nome,
                                "ementa": ementa,
                                "creditos": ch_int // 15 if ch_int > 0 else 0,
                                "carga_horaria": ch_int,
                                "pre_requisito": pre_req,
                                "co_requisito": co_req,
                                "equivalencia": equiva,
                            }

                            buffer_lote.append(materia_dados)

                            # Registra a tupla com parênteses (tupla), não colchetes [lista]
                            codigos_processados.add((codigo, nivel))

                            print(f"  [OK] Processado: {codigo} - {nome} - {nivel}")

                            if len(buffer_lote) >= 10:
                                num += 1
                                salvar_lote_incremental(buffer_lote, CAMINHO_ARQUIVO)
                                buffer_lote.clear()

                            if num == random.randint(len(codigos_processados), len(codigos_processados)+1000):
                                print(f"{len(codigos_processados)} arquivos json foram publicados desde: {inicio_tempo}")
                                num = len(codigos_processados)

                            driver.back()

                        except Exception as e:
                            print(f"  ❌ Erro ao processar item índice {index}: {type(e).__name__}")
                            driver.get(url)
                            break

                except Exception as e:
                    print(f"  [AVISO] Nenhum componente encontrado para: {depto_nome_limpo} ({type(e).__name__})")

    finally:
        if buffer_lote:
            salvar_lote_incremental(buffer_lote, CAMINHO_ARQUIVO)
            buffer_lote.clear()

        driver.quit()
        print(f"\n[SUCESSO] Coleta concluída! Arquivo salvo em: {CAMINHO_ARQUIVO}")
        fim_tempo = time.time()
        hora_fim = datetime.now().strftime("%H:%M:%S")
        duracao_segundos = fim_tempo - inicio_tempo

        print(f"🕒 Início: {hora_inicio} | Fim: {hora_fim} | Duração: {duracao_segundos:.2f}s")


if __name__ == "__main__":
    generate_seed_departamentos()

🚀 Processo iniciado às: 20:22:32
ℹ️ 22 combinações (código, nível) já existem no JSON.
Total de opções encontradas: 5 níveis | 203 departamentos

--- Processando: CAMPUS UNB CEILÂNDIA: FACULDADE DE CIÊNCIAS E TECNOLOGIAS EM SAÚDE (Nível: GRADUAÇÃO) ---
  [OK] Processado: FCE0001 - FUNDAMENTOS DE CITOGENÉTICA HUMANA - GRADUAÇÃO
  [OK] Processado: FCE0002 - INTRODUÇÃO AOS CUIDADOS PALIATIVOS - GRADUAÇÃO
  [OK] Processado: FCE0003 - ESTÁGIO CURRICULAR SUPERVISIONADO EM ENFERMAGEM 2 - GRADUAÇÃO
  [OK] Processado: FCE0004 - INTRODUÇÃO A ANÁLISE E AO APOIO INSTITUCIONAL EM SAÚDE - GRADUAÇÃO
  [OK] Processado: FCE0005 - PATOLOGIA MOLECULAR CLÍNICA - GRADUAÇÃO
  [OK] Processado: FCE0006 - HEMATOLOGIA CLÍNICA - GRADUAÇÃO
  [OK] Processado: FCE0007 - A ENFERMAGEM NA ESTRATÉGIA SAÚDE DA FAMÍLIA - GRADUAÇÃO
  💾 [BACKUP] Lote de 7 itens salvo no JSON em disco!
Caminho: c:\Portfolio\unbook-data-pipeline\data\raw\sigaa\materias.json


[SUCESSO] Coleta concluída! Arquivo salvo em: c:\Portfolio\unbook-

KeyboardInterrupt: 